# **Word Embeddings**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# 📖 TABLE OF CONTENTS

- [Setup](#Setup)
- [Introduction](#Introduction)
- [Represent as Discrete Symbols: One-hot Vectors](#Represent-as-Discrete-Symbols:-One-hot-Vectors)
- [Distributional Semantics](#Distributional-Semantics)
- [References](#References)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Setup

Before we jump in, let's quickly run the setup cell. It checks the environment (GPU, library versions) and sets random seeds for reproducibility — we want identical results on every run.

In [ ]:
# 🔧 Setup: Run this cell first!
# Environment check + random seeds for reproducibility

import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib as mpl
import torch
import torch.nn as nn

print(f"📦 Python       {sys.version.split()[0]}")
print(f"🔢 NumPy        {np.__version__}")
print(f"🐼 Pandas       {pd.__version__}")
print(f"📊 Matplotlib   {mpl.__version__}")
print(f"🔥 PyTorch      {torch.__version__}")

# GPU check (only needed for optional deep-feature experiments)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected")

DEVICE_LABEL = "GPU" if device.type == "cuda" else "CPU"

# Random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\n🎲 Random seed set to {SEED} (Python, NumPy, PyTorch)")

%matplotlib inline
%config InlineBackend.figure_format = 'jpeg'   # photos as JPEG, not PNG — ~10× smaller notebook
# Remove character limits for all columns of Pandas dataframes permanently in this session
pd.set_option('display.max_colwidth', None)

📦 Python       3.13.15
🔢 NumPy        2.1.3
🐼 Pandas       2.2.3
📊 Matplotlib   3.10.0
🔥 PyTorch      2.11.0+cpu
⚠️ No GPU detected

🎲 Random seed set to 42 (Python, NumPy, PyTorch)


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Introduction

The way machine learning models **"see"** data is different from how we (humans) do. For example, we can easily understand the text **"I saw a cat"**, but our models can not - they need vectors of features. Such vectors, or **word embeddings**, are representations of words which can be fed into your model.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">

**How it works: Look-up Table (Vocabulary)**

In practice, you have a vocabulary of allowed words; you choose this vocabulary in advance. For each vocabulary word, a look-up table contains its embedding. This embedding can be found using the word index in the vocabulary (i.e., you to **look up** the embedding in the table using word index).

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-02.gif" width="400">

To account for unknown words (the ones which are not in the vocabulary), usually a vocabulary contains a special token **UNK**. Alternatively, unknown tokens can be ignored or assigned a zero vector.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-03.png" width="300">

**The main question of this lecture is: how do we get these word vectors?**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Represent as Discrete Symbols: One-hot Vectors

The easiest you can do is to represent words as one-hot vectors: for the i-th word in the vocabulary, the vector has 1 on the i-th dimension and 0 on the rest. In Machine Learning, this is the most simple way to represent categorical features.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-04.png" width="250">

You probably can guess why one-hot vectors are not the best way to represent words. One of the problems is that for large vocabularies, these vectors will be very long: vector dimensionality is equal to the vocabulary size. This is undesirable in practice, but this problem is not the most crucial one.

What is really important, is that these vectors **know nothing** about the words they represent. For example, one-hot vectors "think" that cat is as close to dog as it is to table! We can say that **one-hot vectors do not capture meaning**.

**But how do we know what is meaning?**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Distributional Semantics

To capture meaning of words in their vectors, we first need to define the notion of meaning that can be used in practice. For this, let us try to understand how we, humans, get to know which words have similar meaning.

<p float="left">
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-05.png" width="49%" />
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-06.png" width="49%" />
</p>

<p float="left">
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-07.png" width="49%" />
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-08.png" width="49%" />
</p>

<p float="left">
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-09.png" width="49%" />
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-10.png" width="49%" />
</p>

<p float="left">
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-11.png" width="49%" />
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-12.png" width="49%" />
</p>

<p float="left">
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-13.png" width="49%" />
</p>

Once you saw how the unknown word used in different contexts, you were able to understand it's meaning. How did you do this?

The hypothesis is that your brain searched for other words that can be used in the same contexts, found some (e.g., **wine**), and made a conclusion that **tezgüino** has meaning similar to those other words. This is the **distributional hypothesis**:

**Words which frequently appear in similar contexts have similar meaning.**

This is an extremely valuable idea: it can be used in practice to make word vectors capture their meaning. According to the distributional hypothesis, "to capture meaning" and "to capture contexts" are inherently the same. Therefore, all we need to do is to put information about word contexts into word representation.

**Main idea: We need to put information about word contexts into word representation.**

All we'll be doing at this lecture is looking at different ways to do this.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# References

- **[NLP Course by Lena Voita](https://lena-voita.github.io/nlp_course.html)**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)